# Tutorial de Inferência Demográfica

A inferência demográfica fornece informações sobre a dinâmica populacional histórica de uma espécie, incluindo mudanças no tamanho populacional, padrões de migração e eventos de gargalo populacional (*bottlenecks*).

Este tutorial irá focar na análise **PSMC (Pairwise Sequentially Markovian Coalescent)**, um método de inferência demográfica que reconstrói a história do tamanho populacional efetivo (Ne) ao longo do tempo com base em uma única sequência genômica diploide. Desenvolvido por Li e Durbin (2011), o PSMC estima a distribuição do tempo até o ancestral comum mais recente (TMRCA) em diferentes regiões do genoma, modelando variações locais na heterozigosidade sob um processo de coalescência com o modelo Hidden Markov. Isso permite inferir eventos como gargalos, expansões e flutuações demográficas ao longo de centenas de milhares de anos. Na genômica evolutiva e da conservação, o PSMC tem sido amplamente aplicado para reconstruir a história populacional de espécies selvagens, identificar períodos de declínio populacional histórico e comparar trajetórias demográficas entre populações ou espécies, oferecendo insights importantes sobre seus processos evolutivos e resiliência.

Ao final deste tutorial, você será capaz de:

- Compreender as aplicações e limitações do PSMC na inferência demográfica;
- Executar um fluxo completo de análise, desde a preparação dos dados até a visualização dos resultados;
- Interpretar mudanças históricas no tamanho populacional;
- Ajustar parâmetros biológicos, como taxa de mutação e tempo de geração;
- Aplicar informações demográficas em estudos evolutivos e de conservação.


## Seção 0: Preparação do ambiente do Google Colab


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [77]:
# Instalar Miniconda (1–2 min)
import os

# Add conda to the environment
import os
os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']

# Creation of a variable for the base directory
base_dir = "drive/MyDrive/PopGen_UFMG_2026"

miniconda_installer = f"{base_dir}/miniconda/Miniconda3-latest-Linux-x86_64.sh"
if not os.path.exists(miniconda_installer):
    !wget -P "{base_dir}/miniconda" https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

!bash "{miniconda_installer}" -bfp /usr/local


!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [ ]:
# Criação do ambiente Conda e instalação dos programas necessários

!conda create -n genomics_env -c bioconda -c conda-forge -y bcftools psmc gnuplot libwebp

---
## Seção 1: análise

#### Passo 1: obtenha um arquivo `vcf` para um único indivíduo a ser analisado

Vamos utilizar o `bcftools` para criar um arquivo `vcf` para um único indivíduo (BT04).

In [ ]:
sample_id = "BT04"
vcf = "drive/MyDrive/PopGen_UFMG_2026/Material/vcf/bradypus_chr_14_15.vcf.gz"
output_prefix = "drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04"

!conda run -n genomics_env bcftools view -s {sample_id} {vcf} -Oz -o "{output_prefix}.vcf.gz"

Agora vamos indexar o novo `vcf` criado

In [ ]:
!conda run -n genomics_env bcftools index "drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.vcf.gz"

### Passo 2: obtenha a sequência consenso

Utilize o `bcftools` para gerar uma sequência consenso no formato `fasta` a partir de um `vcf`.


In [ ]:
!conda run -n genomics_env bash -c \
'bcftools consensus \
-f "drive/MyDrive/PopGen_UFMG_2026/Material/reference/reference_genome_chr_14_15.fasta" \
"drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.vcf.gz" \
> "drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04_consensus.fasta"'

Vamos entender o que o comando `bcftools consensus` faz. Observe a primeira posição heterozigota do indivíduo `BT04` no seu `vcf`.

In [ ]:
!zgrep -v "^#" drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.vcf.gz | head

A primiera posição varíavel no cromossomo 14 é a 7550.
Agora vamos ver como ficou o arquivo fasta consenso nesta posição.



In [ ]:
!grep -v "^>" "drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04_consensus.fasta" | tr -d '\n' | cut -c7550

Click [aqui](https://www.bioinformatics.org/sms/iupac.html) para ver o código IUPAC.

### Passo 2: Converta FASTA para PSMCFA

Converta o arquivo FASTA consenso para o formato utilizado pelo PSMC (PSMCFA).


In [ ]:
!conda run -n genomics_env fq2psmcfa -q20 \
"drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04_consensus.fasta" > \
"drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.psmcfa"

Explicação:

`-q`: Mantém apenas bases com qualidade igual ou superior a 20.


### Passo 3: Correndo o `psmc`


Iremos correr o PSMC usando o commando:

```
psmc -N25 -t15 -r5 -p "4+25*2+4+6" -o <out>.psmc <entrada>.psmcfa
```

Vamos entender cada parâmetro:

| Parâmetro        | Exemplo          | Significado                                                                 | Observações                                                                 |
|------------------|------------------|------------------------------------------------------------------------------|------------------------------------------------------------------------------|
| `-N <int>`       | `-N25`           | Número máximo de iterações do algoritmo                                     | Valores entre 20 e 30 são comuns. Aumentar pode melhorar a convergência.    |
| `-t <float>`     | `-t15`           | Valor inicial para θ (4Neμ)                                                 | Não afeta o resultado final se o modelo convergir; usado para inicialização.|
| `-r <float>`     | `-r5`            | Razão inicial entre recombinação e mutação (ρ/θ)                            | Um valor razoável ajuda o modelo a se ajustar mais rápido.                  |
| `-p <string>`    | `-p "4+25*2+4+6"`| Modelo de discretização do tempo                                            | Controla a resolução temporal do modelo; afeta sensibilidade a eventos.     |
| `-o <arquivo>`   | `-o saida.psmc`  | Nome do arquivo de saída `.psmc`                                            | Será usado como entrada no `psmc_plot.pl`.                                  |
| `<entrada>`      | `entrada.psmcfa` | Arquivo `.psmcfa` gerado a partir da sequência consenso (`fq2psmcfa`)       | Deve ser preparado corretamente com base no genoma e VCF do indivíduo.      |


---
#### 📌 Explicando o parâmetro `-p "4+25*2+4+6"`

Este parâmetro define como a linha do tempo será dividida para estimar o tamanho populacional.


| Parte        | Interpretação                              |
|--------------|---------------------------------------------|
| `4`          | 4 intervalos para tempos recentes           |
| `25*2`       | 25 blocos com 2 intervalos cada (50 total) |
| `4`          | 4 intervalos intermediários                 |
| `6`          | 6 intervalos para tempos antigos            |



Você pode ajustar o parâmetro `-p` de acordo com o foco da sua análise:


| p            | Objetivo       |Interpretação     |
|--------------|----------------|------------------|
|-p "10+15*2+4+4"|Foco em eventos recentes|Aumenta a resolução no tempo recente. Ideal para detectar gargalos populacionais associados a eventos modernos (como impactos humanos).|
|-p "4+20*2+10+10"|Foco no passado profundo| Aumenta a resolução em tempos antigos. Útil para estudar divergência entre populações ou espécies, ou eventos do Pleistoceno.|
|-p "4+10*2+2+2"|Execução rápida para testes|Modelo mais leve, com menos bins. Útil para testes preliminares, especialmente em sequências consenso curtas ou simulações.|


🌍 Analogia - Imagine observar um mapa:

- Tempos recentes → zoom máximo → maior detalhe
- Tempos antigos → zoom reduzido → menor detalhe

Isso permite que o PSMC utilize maior resolução onde os dados são mais informativos.


⚠️ Dicas importantes
* O número total de intervalos deve ser ≤ 64, conforme a recomendação do próprio PSMC.
* Use mais intervalos onde você espera eventos demográficos importantes (por exemplo, nos tempos recentes para gargalos).
* Evite usar muitos intervalos em regiões do genoma com cobertura baixa ou ruído alto, pois isso pode gerar curvas instáveis.
* Sempre verifique o gráfico final (com `psmc_plot.pl`) para avaliar se o modelo de tempo escolhido capturou os padrões esperados.

---



In [ ]:
!conda run -n genomics_env psmc -N25 -t15 -r5 -p "4+10*2+2+2" \
    drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.psmcfa \
    > drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.psmc

### Passo 4: Plotando o resultado

Iremos plotar o resultado usando o sequinte comando:
```
psmc_plot.pl -g <generation_time> -u <mutation_rate> <out> <prefix>.psmc
```

---
#### 📌 Escolhendo a taxa de mutação (`-u`) e o tempo de geração (`-g`) no PSMC

Os parâmetros `-u` e `-g` são usados **somente na etapa de visualização** (`psmc_plot.pl`) para converter as estimativas do PSMC — que estão em **unidades coalescentes relativas** — em **escalas reais de tempo e tamanho populacional**.

* **`-u`: Taxa de mutação por base por geração**

A taxa de mutação determina como o tempo e o tamanho populacional serão **escalados para unidades reais** (como anos e número de indivíduos). Esse valor deve refletir a **taxa média de substituições por base por geração** para a espécie (ou grupo próximo) que você está estudando.

| Grupo                   | Valor típico de `-u`      | Fonte / Comentário                                  |
|-------------------------|----------------------------|-----------------------------------------------------|
| Humanos                 | `1.25e-8` a `1.5e-8`       | Medido empiricamente em trios familiares            |
| Mamíferos não humanos   | `1e-8`                     | Valor conservador amplamente utilizado              |
| Aves                    | `2e-9` a `4e-9`            | Menor taxa de mutação; usar dados de literatura     |
| Répteis / Anfíbios      | `1e-9` a `2e-9`            | Taxas mais lentas; dependem do grupo                |
| Peixes                  | `5e-9` a `1e-8`            | Pode variar bastante                                |
| Insetos                 | `2e-9` a `8e-9`            | Estudos com Drosophila são comuns como referência   |

> Uma prática comum quando não se conhece a taxa exata para a espécie estudada é utilizar um valor estimado para o grupo mais próximo e citar a fonte na sua análise.

* **`-g`: Tempo médio de geração (em anos)**

Este parâmetro representa o tempo médio **entre o nascimento de um indivíduo e o nascimento de sua prole**. Ele é essencial para converter os tempos inferidos pelo PSMC de "número de gerações" para "milhares de anos".


> Use dados ecológicos, literatura populacional ou bancos como PanTHERIA, AnAge, BirdLife ou Animal Diversity Web para obter o tempo de geração.

* Como afetam o gráfico PSMC?

| Parâmetro | Impacto no gráfico PSMC                     |
|-----------|---------------------------------------------|
| `-u`      | Escala o eixo **Y** (tamanho efetivo, Ne)    |
| `-g`      | Escala o eixo **X** (tempo, em mil anos)     |


Se você dobrar `-g`, os eventos aparecerão o dobro de tempo no passado; se dobrar `-u`, o Ne estimado será reduzido pela metade.

---

In [ ]:
!conda create -n psmc_clean -c conda-forge \
  gnuplot \
  libwebp \
  perl \
  perl-app-cpanminus \
  texlive-core \
  ghostscript

In [ ]:
!conda run -n genomics_env psmc_plot.pl -g 4 -u 1e-8 \
     -p drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04 \
     drive/MyDrive/PopGen_UFMG_2026/analyses/09_PSMC/BT04.psmc

### Passo 5: Interpretando o Gráfico do PSMC

O gráfico possui dois eixos principais. O eixo X representa o tempo antes do presente (Esquerda → mais recente; Direita → mais antigo). O eixo Y apresenta o tamanho populacional efetivo (Ne), representando uma estimativa do número efetivo de indivíduos reprodutivos ao longo do tempo. Picos indicam expansão populacional, que podem estar associados a diversos eventos, como condições ambientais favoráveis,  expansão geográfica e redução da competição. Vales Indicam gargalos populacionais (*bottlenecks*), possivelmente associados a perda de habitat, mudanças climáticas e atividades humanas. Regiões Planas indicam estabilidade populacional durante longos períodos.


> **🦥❓ Questões:**
>
>1. O que acontece com o gráfico quando a taxa de mutação é alterada?
>2. Como a mudança do tempo de geração afeta a interpretação temporal dos resultados?
>3. O que a trajetória do PSMC sugere sobre a história evolutiva das preguiças de coleira?
>4. Quais recomendações de conservação podem ser feitas com base nos resultados?
>5. Quais análises adicionais poderiam fornecer inferências demográficas mais recentes e detalhadas?
